# 03 — Model Selection

Compare LightGBM, XGBoost, and CatBoost on the validation set.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from training.preprocessing import build_preprocessor, engineer_features, ALL_FEATURES

TARGET = 'log_resale_price'

df = engineer_features(pd.read_csv('../../data/hdb_resale_before_2025.csv'))
train = df[df['transaction_year'] <= 2024]
val   = df[(df['transaction_year'] == 2025) & (df['transaction_month'] <= 9)]

preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(train[ALL_FEATURES], train[TARGET])
X_val   = preprocessor.transform(val[ALL_FEATURES])
y_train, y_val = train[TARGET], val[TARGET]

In [ ]:
def evaluate(name, model, X, y_log):
    pred = np.expm1(model.predict(X))
    true = np.expm1(y_log)
    mae  = mean_absolute_error(true, pred)
    rmse = float(np.sqrt(mean_squared_error(true, pred)))
    mape = float(np.mean(np.abs((true - pred) / true)) * 100)
    r2   = r2_score(true, pred)
    print(f'{name:12s}: MAE={mae:,.0f} | RMSE={rmse:,.0f} | MAPE={mape:.2f}% | R²={r2:.4f}')
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

In [ ]:
# LightGBM
lgbm = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1
)
lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)],
         callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
res_lgbm = evaluate('LightGBM', lgbm, X_val, y_val)

In [ ]:
# XGBoost
xgbm = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='rmse', early_stopping_rounds=50, random_state=42
)
xgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
res_xgb = evaluate('XGBoost', xgbm, X_val, y_val)

In [ ]:
# Summary
import pandas as pd
results = pd.DataFrame([res_lgbm, res_xgb])
results.set_index('model').sort_values('MAE')